# Airfare Analysis

This notebook analyzes airfare for the Top 10 domestic destinations from Seattle-Tacoma International Airport (SEA).

Airfare data comes from two BTS Origin and Destination Survey datasets:

- **DB1B Market:** Q4 2022, Q4 2023, and Q4 2024
- **DB1C Market:** December 2025

Because DB1B and DB1C use different reporting periods and sampling methodologies, the airfare values are used primarily for descriptive comparison rather than as a perfectly equivalent four-year fare series.

## 1. Setup and Data Sources

Required libraries and file paths are initialized for the airfare analysis. The Top 10 destination airports identified in the T-100 demand analysis are used throughout this notebook.

In [1]:
from pathlib import Path
import zipfile
import pandas as pd

In [2]:
raw_folder = Path("../Data/raw")

airfare_files = sorted(
    list(raw_folder.glob("*DB1BMarket*.zip")) +
    list(raw_folder.glob("DB1C.MARKET*.zip"))
)

for file in airfare_files:
    print(file.name)

DB1C.MARKET.202512.27MAY2026.zip
Origin_and_Destination_Survey_DB1BMarket_2022_4.zip
Origin_and_Destination_Survey_DB1BMarket_2023_4.zip
Origin_and_Destination_Survey_DB1BMarket_2024_4.zip


## 2. Process DB1B Market Data

DB1B Market provides quarterly Origin and Destination Survey data based on a 10% sample of airline tickets.

For 2022–2024, Q4 records are filtered to:

- Origin airport = `SEA`
- Top 10 destination airports
- One-coupon markets (`MktCoupons = 1`) to represent nonstop markets
- Positive market fares (`MktFare > 0`)

Passenger-weighted average fares are then calculated for each destination.

### Q4 2022

In [3]:
db1b_2022_file = next(
    file for file in airfare_files
    if "DB1BMarket_2022_4" in file.name
)

with zipfile.ZipFile(db1b_2022_file, "r") as z:
    print(db1b_2022_file.name)
    print(z.namelist())

Origin_and_Destination_Survey_DB1BMarket_2022_4.zip
['Origin_and_Destination_Survey_DB1BMarket_2022_4.csv', 'readme.html']


In [4]:
with zipfile.ZipFile(db1b_2022_file, "r") as z:
    csv_name = [
        name for name in z.namelist()
        if name.endswith(".csv")
    ][0]

    db1b_columns = pd.read_csv(
        z.open(csv_name),
        nrows=0
    ).columns.tolist()

db1b_columns

['ItinID',
 'MktID',
 'MktCoupons',
 'Year',
 'Quarter',
 'OriginAirportID',
 'OriginAirportSeqID',
 'OriginCityMarketID',
 'Origin',
 'OriginCountry',
 'OriginStateFips',
 'OriginState',
 'OriginStateName',
 'OriginWac',
 'DestAirportID',
 'DestAirportSeqID',
 'DestCityMarketID',
 'Dest',
 'DestCountry',
 'DestStateFips',
 'DestState',
 'DestStateName',
 'DestWac',
 'AirportGroup',
 'WacGroup',
 'TkCarrierChange',
 'TkCarrierGroup',
 'OpCarrierChange',
 'OpCarrierGroup',
 'RPCarrier',
 'TkCarrier',
 'OpCarrier',
 'BulkFare',
 'Passengers',
 'MktFare',
 'MktDistance',
 'MktDistanceGroup',
 'MktMilesFlown',
 'NonStopMiles',
 'ItinGeoType',
 'MktGeoType',
 'Unnamed: 41']

In [5]:
fare_columns = [
    "Year",
    "Quarter",
    "OriginAirportID",
    "Origin",
    "OriginState",
    "DestAirportID",
    "Dest",
    "DestState",
    "Passengers",
    "MktFare",
    "MktCoupons",
    "RPCarrier"
]

fare_columns

['Year',
 'Quarter',
 'OriginAirportID',
 'Origin',
 'OriginState',
 'DestAirportID',
 'Dest',
 'DestState',
 'Passengers',
 'MktFare',
 'MktCoupons',
 'RPCarrier']

In [6]:
sea_chunks = []

with zipfile.ZipFile(db1b_2022_file, "r") as z:
    for chunk in pd.read_csv(
        z.open(csv_name),
        usecols=fare_columns,
        chunksize=200000
    ):
        sea_chunk = chunk[chunk["Origin"] == "SEA"]
        
        if not sea_chunk.empty:
            sea_chunks.append(sea_chunk)

db1b_2022_sea = pd.concat(sea_chunks, ignore_index=True)

db1b_2022_sea.shape

(137236, 12)

In [7]:
print("Years:")
print(db1b_2022_sea["Year"].value_counts())

print("\nQuarters:")
print(db1b_2022_sea["Quarter"].value_counts())

print("\nOrigins:")
print(db1b_2022_sea["Origin"].value_counts())

Years:
Year
2022    137236
Name: count, dtype: int64

Quarters:
Quarter
4    137236
Name: count, dtype: int64

Origins:
Origin
SEA    137236
Name: count, dtype: int64


In [8]:
top10_airports = [
    "LAX", "PHX", "LAS", "ANC", "DEN",
    "SFO", "DFW", "ORD", "SAN", "PDX"
]

top10_airports

['LAX', 'PHX', 'LAS', 'ANC', 'DEN', 'SFO', 'DFW', 'ORD', 'SAN', 'PDX']

In [9]:
db1b_2022_top10 = db1b_2022_sea[
    db1b_2022_sea["Dest"].isin(top10_airports)
].copy()

db1b_2022_top10.shape

(31307, 12)

In [10]:
db1b_2022_top10["MktFare"].describe()

count    31307.00000
mean       238.26825
std        157.96909
min          0.00000
25%        136.00000
50%        208.50000
75%        302.69500
max       2274.00000
Name: MktFare, dtype: float64

In [11]:
zero_fares = db1b_2022_top10[
    db1b_2022_top10["MktFare"] <= 0
]

zero_fares.shape

(37, 12)

In [12]:
db1b_2022_top10["MktCoupons"].value_counts().sort_index()

MktCoupons
1    27172
2     3881
3      245
4        7
5        1
6        1
Name: count, dtype: int64

In [13]:
db1b_2022_clean = db1b_2022_top10[
    (db1b_2022_top10["MktFare"] > 0) &
    (db1b_2022_top10["MktCoupons"] == 1)
].copy()

db1b_2022_clean.shape

(27138, 12)

In [14]:
db1b_2022_clean["Passengers"].describe()

count    27138.000000
mean         3.765642
std         11.923099
min          1.000000
25%          1.000000
50%          1.000000
75%          2.000000
max        539.000000
Name: Passengers, dtype: float64

In [15]:
db1b_2022_clean["Passengers"].value_counts().sort_index().head(20)

Passengers
1.0     16689
2.0      4003
3.0      1762
4.0      1007
5.0       651
6.0       475
7.0       329
8.0       212
9.0       203
10.0      159
11.0      118
12.0      128
13.0      106
14.0       70
15.0       88
16.0       74
17.0       58
18.0       56
19.0       55
20.0       45
Name: count, dtype: int64

In [16]:
fare_2022 = (
    db1b_2022_clean
    .assign(
        FARE_X_PASSENGERS=
        db1b_2022_clean["MktFare"] *
        db1b_2022_clean["Passengers"]
    )
    .groupby("Dest", as_index=False)
    .agg(
        PASSENGERS=("Passengers", "sum"),
        FARE_X_PASSENGERS=("FARE_X_PASSENGERS", "sum")
    )
)

fare_2022["AVG_FARE"] = (
    fare_2022["FARE_X_PASSENGERS"] /
    fare_2022["PASSENGERS"]
).round(2)

fare_2022 = fare_2022[
    ["Dest", "PASSENGERS", "AVG_FARE"]
].sort_values("AVG_FARE")

fare_2022

,Dest,PASSENGERS,AVG_FARE
3,LAS,16284.0,147.90
9,SFO,12521.0,154.75
1,DEN,10886.0,163.52
6,PDX,2093.0,178.91
8,SAN,11503.0,182.47
7,PHX,15499.0,191.28
4,LAX,14387.0,205.28
0,ANC,3989.0,219.94
5,ORD,8663.0,223.18
2,DFW,6367.0,263.67


### Q4 2023

In [17]:
db1b_2023_file = next(
    file for file in airfare_files
    if "DB1BMarket_2023_4" in file.name
)

db1b_2023_file

WindowsPath('../Data/raw/Origin_and_Destination_Survey_DB1BMarket_2023_4.zip')

In [18]:
chunks_2023 = []

with zipfile.ZipFile(db1b_2023_file, "r") as z:
    csv_name_2023 = [
        name for name in z.namelist()
        if name.endswith(".csv")
    ][0]

    for chunk in pd.read_csv(
        z.open(csv_name_2023),
        usecols=fare_columns,
        chunksize=200000
    ):
        filtered_chunk = chunk[
            (chunk["Origin"] == "SEA") &
            (chunk["Dest"].isin(top10_airports))
        ]

        if not filtered_chunk.empty:
            chunks_2023.append(filtered_chunk)

db1b_2023_top10 = pd.concat(
    chunks_2023,
    ignore_index=True
)

db1b_2023_top10.shape

(33152, 12)

In [19]:
print("Years:")
print(db1b_2023_top10["Year"].value_counts())

print("\nQuarters:")
print(db1b_2023_top10["Quarter"].value_counts())

print("\nOrigins:")
print(db1b_2023_top10["Origin"].value_counts())

print("\nDestinations:")
print(db1b_2023_top10["Dest"].value_counts())

Years:
Year
2023    33152
Name: count, dtype: int64

Quarters:
Quarter
4    33152
Name: count, dtype: int64

Origins:
Origin
SEA    33152
Name: count, dtype: int64

Destinations:
Dest
PHX    4848
LAX    4485
LAS    4155
SFO    4122
DEN    3761
ORD    3497
SAN    2999
DFW    2758
PDX    1342
ANC    1185
Name: count, dtype: int64


In [20]:
db1b_2023_clean = db1b_2023_top10[
    (db1b_2023_top10["MktFare"] > 0) &
    (db1b_2023_top10["MktCoupons"] == 1)
].copy()

db1b_2023_clean.shape

(29868, 12)

In [21]:
fare_2023 = (
    db1b_2023_clean
    .assign(
        FARE_X_PASSENGERS=
        db1b_2023_clean["MktFare"] *
        db1b_2023_clean["Passengers"]
    )
    .groupby("Dest", as_index=False)
    .agg(
        PASSENGERS=("Passengers", "sum"),
        FARE_X_PASSENGERS=("FARE_X_PASSENGERS", "sum")
    )
)

fare_2023["AVG_FARE"] = (
    fare_2023["FARE_X_PASSENGERS"] /
    fare_2023["PASSENGERS"]
).round(2)

fare_2023 = fare_2023[
    ["Dest", "PASSENGERS", "AVG_FARE"]
].sort_values("AVG_FARE")

fare_2023

,Dest,PASSENGERS,AVG_FARE
3,LAS,16601.0,139.33
4,LAX,16805.0,149.98
7,PHX,16735.0,158.87
1,DEN,10994.0,161.81
6,PDX,2137.0,166.86
9,SFO,12604.0,169.95
8,SAN,10821.0,171.90
5,ORD,9627.0,201.64
0,ANC,3707.0,205.09
2,DFW,7424.0,242.63


In [22]:
fare_comparison = (
    fare_2022[["Dest", "AVG_FARE"]]
    .rename(columns={"AVG_FARE": "FARE_2022"})
    .merge(
        fare_2023[["Dest", "AVG_FARE"]]
        .rename(columns={"AVG_FARE": "FARE_2023"}),
        on="Dest"
    )
)

fare_comparison["CHANGE_PCT"] = (
    (fare_comparison["FARE_2023"] - fare_comparison["FARE_2022"])
    / fare_comparison["FARE_2022"]
    * 100
).round(1)

fare_comparison.sort_values("CHANGE_PCT")

,Dest,FARE_2022,FARE_2023,CHANGE_PCT
6,LAX,205.28,149.98,-26.9
5,PHX,191.28,158.87,-16.9
8,ORD,223.18,201.64,-9.7
9,DFW,263.67,242.63,-8.0
7,ANC,219.94,205.09,-6.8
3,PDX,178.91,166.86,-6.7
0,LAS,147.90,139.33,-5.8
4,SAN,182.47,171.90,-5.8
2,DEN,163.52,161.81,-1.0
1,SFO,154.75,169.95,9.8


### Q4 2024

In [23]:
db1b_2024_file = next(
    file for file in airfare_files
    if "DB1BMarket_2024_4" in file.name
)

db1b_2024_file

WindowsPath('../Data/raw/Origin_and_Destination_Survey_DB1BMarket_2024_4.zip')

In [24]:
chunks_2024 = []

with zipfile.ZipFile(db1b_2024_file, "r") as z:
    csv_name_2024 = [
        name for name in z.namelist()
        if name.endswith(".csv")
    ][0]

    for chunk in pd.read_csv(
        z.open(csv_name_2024),
        usecols=fare_columns,
        chunksize=200000
    ):
        filtered_chunk = chunk[
            (chunk["Origin"] == "SEA") &
            (chunk["Dest"].isin(top10_airports))
        ]

        if not filtered_chunk.empty:
            chunks_2024.append(filtered_chunk)

db1b_2024_top10 = pd.concat(
    chunks_2024,
    ignore_index=True
)

db1b_2024_top10.shape

(36549, 12)

In [25]:
print("Years:")
print(db1b_2024_top10["Year"].value_counts())

print("\nQuarters:")
print(db1b_2024_top10["Quarter"].value_counts())

print("\nOrigins:")
print(db1b_2024_top10["Origin"].value_counts())

print("\nDestinations:")
print(db1b_2024_top10["Dest"].value_counts())

Years:
Year
2024    36549
Name: count, dtype: int64

Quarters:
Quarter
4    36549
Name: count, dtype: int64

Origins:
Origin
SEA    36549
Name: count, dtype: int64

Destinations:
Dest
PHX    5285
SFO    4641
DEN    4466
LAX    4439
LAS    4356
ORD    4213
DFW    3402
SAN    3107
ANC    1378
PDX    1262
Name: count, dtype: int64


In [26]:
db1b_2024_clean = db1b_2024_top10[
    (db1b_2024_top10["MktFare"] > 0) &
    (db1b_2024_top10["MktCoupons"] == 1)
].copy()

db1b_2024_clean.shape

(33362, 12)

In [27]:
fare_2024 = (
    db1b_2024_clean
    .assign(
        FARE_X_PASSENGERS=
        db1b_2024_clean["MktFare"] *
        db1b_2024_clean["Passengers"]
    )
    .groupby("Dest", as_index=False)
    .agg(
        PASSENGERS=("Passengers", "sum"),
        FARE_X_PASSENGERS=("FARE_X_PASSENGERS", "sum")
    )
)

fare_2024["AVG_FARE"] = (
    fare_2024["FARE_X_PASSENGERS"] /
    fare_2024["PASSENGERS"]
).round(2)

fare_2024 = fare_2024[
    ["Dest", "PASSENGERS", "AVG_FARE"]
].sort_values("AVG_FARE")

fare_2024

,Dest,PASSENGERS,AVG_FARE
3,LAS,16440.0,144.77
6,PDX,2185.0,152.47
1,DEN,11465.0,163.03
7,PHX,17579.0,164.62
4,LAX,17146.0,169.38
8,SAN,11886.0,175.79
9,SFO,13576.0,177.04
2,DFW,8992.0,207.70
5,ORD,10085.0,217.43
0,ANC,4132.0,263.05


In [28]:
fare_2022_2024 = (
    fare_2022[["Dest", "AVG_FARE"]]
    .rename(columns={"AVG_FARE": "FARE_2022"})
    .merge(
        fare_2023[["Dest", "AVG_FARE"]]
        .rename(columns={"AVG_FARE": "FARE_2023"}),
        on="Dest"
    )
    .merge(
        fare_2024[["Dest", "AVG_FARE"]]
        .rename(columns={"AVG_FARE": "FARE_2024"}),
        on="Dest"
    )
)

fare_2022_2024

,Dest,FARE_2022,FARE_2023,FARE_2024
0,LAS,147.90,139.33,144.77
1,SFO,154.75,169.95,177.04
2,DEN,163.52,161.81,163.03
3,PDX,178.91,166.86,152.47
4,SAN,182.47,171.90,175.79
5,PHX,191.28,158.87,164.62
6,LAX,205.28,149.98,169.38
7,ANC,219.94,205.09,263.05
8,ORD,223.18,201.64,217.43
9,DFW,263.67,242.63,207.70


## 3. Process December 2025 DB1C Market Data

December 2025 airfare is analyzed using DB1C Market, which provides monthly Origin and Destination Survey data using a 40% sample of tickets.

Records are filtered to:

- Reporting year = 2025
- Reporting month = December
- Origin airport = `SEA`
- Top 10 destination airports
- Nonstop markets (`Nonstop = 1`)
- Positive market amounts (`MktAmount > 0`)

The analyzed DB1C records contain one passenger per record, so the passenger-weighted mean and simple mean produce the same result.

In [29]:
db1c_2025_file = next(
    file for file in airfare_files
    if "DB1C.MARKET" in file.name
)

db1c_2025_file

WindowsPath('../Data/raw/DB1C.MARKET.202512.27MAY2026.zip')

In [30]:
with zipfile.ZipFile(db1c_2025_file, "r") as z:
    db1c_contents = z.namelist()

db1c_contents

['DB1C.MARKET.202512.27MAY2026.asc.zip',
 'DB1C.MARKET.202512.27MAY2026.csv.zip',
 'DB1C.MARKET.202512.27MAY2026.parquet']

In [31]:
import pyarrow.parquet as pq

with zipfile.ZipFile(db1c_2025_file, "r") as z:
    parquet_name = [
        name for name in z.namelist()
        if name.lower().endswith(".parquet")
    ][0]

    with z.open(parquet_name) as parquet_file:
        parquet_data = parquet_file.read()

from io import BytesIO

db1c_parquet = BytesIO(parquet_data)

parquet_schema = pq.read_schema(db1c_parquet)

db1c_columns = parquet_schema.names

db1c_columns

['ItinID',
 'MktID',
 'GateID',
 'MktCoupons',
 'RpYear',
 'RpQuarter',
 'RpMonth',
 'SchFlYear',
 'SchFlQuarter',
 'SchFlMonth',
 'OriginAirportID',
 'OriginAirportSeqID',
 'OriginCityMarketID',
 'Origin',
 'OriginCountry',
 'OriginStateFips',
 'OriginState',
 'OriginStateName',
 'OriginWac',
 'DestAirportID',
 'DestAirportSeqID',
 'DestCityMarketID',
 'Dest',
 'DestCountry',
 'DestStateFips',
 'DestState',
 'DestStateName',
 'DestWac',
 'AirportGroup',
 'WacGroup',
 'DwellTimeGroup',
 'SegmentViaGroup',
 'RpCarrierAirlineID',
 'RpCarrier',
 'IssuingCarrierAirlineID',
 'IssuingCarrier',
 'MktCarrierAirlineID',
 'MktCarrier',
 'MktCarrierGroup',
 'MktCarrierChange',
 'OpCarrierAirlineID',
 'OpCarrier',
 'OpCarrierGroup',
 'OpCarrierChange',
 'Passengers',
 'MktAmount',
 'MktTax',
 'V_Yield',
 'PurchaseWindowGroup',
 'TotalDistance',
 'MilesTraveled',
 'NonStopMiles',
 'MktDistanceGroup',
 'Nonstop',
 'MktGeoType',
 'ItinGeoType',
 'Break_LegacyLogic']

In [32]:
db1c_extract_path = raw_folder / parquet_name

if not db1c_extract_path.exists():
    with zipfile.ZipFile(db1c_2025_file, "r") as z:
        z.extract(parquet_name, raw_folder)

db1c_extract_path

WindowsPath('../Data/raw/DB1C.MARKET.202512.27MAY2026.parquet')

In [33]:
db1c_fare_columns = [
    "RpYear",
    "RpQuarter",
    "RpMonth",
    "OriginAirportID",
    "Origin",
    "OriginState",
    "DestAirportID",
    "Dest",
    "DestState",
    "Passengers",
    "MktAmount",
    "MktCoupons",
    "Nonstop",
    "RpCarrier"
]

db1c_fare_columns

['RpYear',
 'RpQuarter',
 'RpMonth',
 'OriginAirportID',
 'Origin',
 'OriginState',
 'DestAirportID',
 'Dest',
 'DestState',
 'Passengers',
 'MktAmount',
 'MktCoupons',
 'Nonstop',
 'RpCarrier']

In [34]:
db1c_2025 = pd.read_parquet(
    db1c_extract_path,
    columns=db1c_fare_columns
)

db1c_2025.shape

(21359385, 14)

In [35]:
db1c_2025_top10 = db1c_2025[
    (db1c_2025["Origin"] == "SEA") &
    (db1c_2025["Dest"].isin(top10_airports))
].copy()

db1c_2025_top10.shape

(142801, 14)

In [36]:
print("Years:")
print(db1c_2025_top10["RpYear"].value_counts())

print("\nQuarters:")
print(db1c_2025_top10["RpQuarter"].value_counts())

print("\nMonths:")
print(db1c_2025_top10["RpMonth"].value_counts())

print("\nOrigins:")
print(db1c_2025_top10["Origin"].value_counts())

print("\nDestinations:")
print(db1c_2025_top10["Dest"].value_counts())

Years:
RpYear
2025    142801
Name: count, dtype: int64

Quarters:
RpQuarter
4    142801
Name: count, dtype: int64

Months:
RpMonth
12    142801
Name: count, dtype: int64

Origins:
Origin
SEA    142801
Name: count, dtype: int64

Destinations:
Dest
PHX    23139
LAX    20676
LAS    18542
SFO    17603
SAN    15915
DEN    14583
DFW    13325
ORD    11498
ANC     4771
PDX     2749
Name: count, dtype: int64


In [37]:
db1c_2025_top10["Nonstop"].value_counts(dropna=False)

Nonstop
1    139352
0      3449
Name: count, dtype: int64

In [38]:
db1c_2025_top10["MktAmount"].describe()

count    142798.000000
mean        199.881622
std         136.038073
min           0.000000
25%         105.800000
50%         179.300000
75%         273.300000
max        2528.690000
Name: MktAmount, dtype: float64

In [39]:
print("Missing:")
print(db1c_2025_top10["MktAmount"].isna().sum())

print("\nZero or negative:")
print((db1c_2025_top10["MktAmount"] <= 0).sum())

print("\nNonstop records:")
print((db1c_2025_top10["Nonstop"] == 1).sum())

Missing:
3

Zero or negative:
743

Nonstop records:
139352


In [40]:
print(db1c_2025_top10["Passengers"].describe())

print("\nPassenger values:")
print(
    db1c_2025_top10["Passengers"]
    .value_counts()
    .sort_index()
    .head(20)
)

count    142801.0
mean          1.0
std           0.0
min           1.0
25%           1.0
50%           1.0
75%           1.0
max           1.0
Name: Passengers, dtype: float64

Passenger values:
Passengers
1.0    142801
Name: count, dtype: int64


In [41]:
db1c_2025_clean = db1c_2025_top10[
    (db1c_2025_top10["MktAmount"].notna()) &
    (db1c_2025_top10["MktAmount"] > 0) &
    (db1c_2025_top10["Nonstop"] == 1)
].copy()

db1c_2025_clean.shape

(138606, 14)

In [42]:
fare_2025 = (
    db1c_2025_clean
    .assign(
        FARE_X_PASSENGERS=
        db1c_2025_clean["MktAmount"] *
        db1c_2025_clean["Passengers"]
    )
    .groupby("Dest", as_index=False)
    .agg(
        PASSENGERS=("Passengers", "sum"),
        FARE_X_PASSENGERS=("FARE_X_PASSENGERS", "sum")
    )
)

fare_2025["AVG_FARE"] = (
    fare_2025["FARE_X_PASSENGERS"] /
    fare_2025["PASSENGERS"]
).round(2)

fare_2025 = fare_2025[
    ["Dest", "PASSENGERS", "AVG_FARE"]
].sort_values("AVG_FARE")

fare_2025

,Dest,PASSENGERS,AVG_FARE
3,LAS,18261.0,170.65
1,DEN,14352.0,173.53
7,PHX,22573.0,185.09
6,PDX,2228.0,191.22
9,SFO,17474.0,192.84
4,LAX,20173.0,206.81
8,SAN,14516.0,221.54
2,DFW,12998.0,227.33
5,ORD,11287.0,233.31
0,ANC,4744.0,282.22


## 4. Combine Airfare Results

The destination-level airfare results from Q4 2022–2024 DB1B and December 2025 DB1C are combined into a single comparison table.

A long-format version is also created for use in Power BI. The source period is retained so that the DB1B Q4 values and DB1C December 2025 values remain clearly distinguished.

In [43]:
fare_2022_2025 = (
    fare_2022_2024
    .merge(
        fare_2025[["Dest", "AVG_FARE"]]
        .rename(columns={"AVG_FARE": "FARE_2025"}),
        on="Dest",
        how="inner"
    )
)

fare_2022_2025

,Dest,FARE_2022,FARE_2023,FARE_2024,FARE_2025
0,LAS,147.90,139.33,144.77,170.65
1,SFO,154.75,169.95,177.04,192.84
2,DEN,163.52,161.81,163.03,173.53
3,PDX,178.91,166.86,152.47,191.22
4,SAN,182.47,171.90,175.79,221.54
5,PHX,191.28,158.87,164.62,185.09
6,LAX,205.28,149.98,169.38,206.81
7,ANC,219.94,205.09,263.05,282.22
8,ORD,223.18,201.64,217.43,233.31
9,DFW,263.67,242.63,207.70,227.33


In [44]:
fare_long = fare_2022_2025.melt(
    id_vars="Dest",
    var_name="PERIOD",
    value_name="AVG_FARE"
)

fare_long["YEAR"] = (
    fare_long["PERIOD"]
    .str.extract(r"(\d{4})")
    .astype(int)
)

fare_long["SOURCE"] = fare_long["YEAR"].apply(
    lambda year: "DB1B Q4" if year <= 2024 else "DB1C December"
)

fare_long = fare_long[
    ["Dest", "YEAR", "SOURCE", "AVG_FARE"]
].sort_values(["Dest", "YEAR"])

fare_long

,Dest,YEAR,SOURCE,AVG_FARE
7,ANC,2022,DB1B Q4,219.94
17,ANC,2023,DB1B Q4,205.09
27,ANC,2024,DB1B Q4,263.05
37,ANC,2025,DB1C December,282.22
2,DEN,2022,DB1B Q4,163.52
12,DEN,2023,DB1B Q4,161.81
22,DEN,2024,DB1B Q4,163.03
32,DEN,2025,DB1C December,173.53
9,DFW,2022,DB1B Q4,263.67
19,DFW,2023,DB1B Q4,242.63


## 5. Validate Final Airfare Datasets

The final datasets are checked for destination coverage, missing airfare values, and expected table dimensions before export.

In [45]:
print("Number of destinations:")
print(fare_2022_2025["Dest"].nunique())

print("\nDestinations:")
print(sorted(fare_2022_2025["Dest"].tolist()))

print("\nMissing fare values:")
print(fare_2022_2025.isna().sum())

print("\nLong table shape:")
print(fare_long.shape)

Number of destinations:
10

Destinations:
['ANC', 'DEN', 'DFW', 'LAS', 'LAX', 'ORD', 'PDX', 'PHX', 'SAN', 'SFO']

Missing fare values:
Dest         0
FARE_2022    0
FARE_2023    0
FARE_2024    0
FARE_2025    0
dtype: int64

Long table shape:
(40, 4)


## 6. Export Processed Airfare Data

The final wide-format and long-format airfare datasets are exported as CSV files for use in the master analysis and Power BI dashboard.

In [46]:
processed_folder = Path("../Data/processed")
processed_folder.mkdir(parents=True, exist_ok=True)

fare_2022_2025.to_csv(
    processed_folder / "top10_airfare_2022_2025.csv",
    index=False
)

fare_long.to_csv(
    processed_folder / "top10_airfare_long_2022_2025.csv",
    index=False
)

print("Saved successfully.")

Saved successfully.


In [47]:
list(processed_folder.glob("*.csv"))

[WindowsPath('../Data/processed/airline_by_destination_top10.csv'),
 WindowsPath('../Data/processed/airline_service_top10.csv'),
 WindowsPath('../Data/processed/master_top10_destinations.csv'),
 WindowsPath('../Data/processed/state_demand.csv'),
 WindowsPath('../Data/processed/top10_2025_balance.csv'),
 WindowsPath('../Data/processed/top10_airfare_2022_2025.csv'),
 WindowsPath('../Data/processed/top10_airfare_long_2022_2025.csv'),
 WindowsPath('../Data/processed/top10_yearly_demand.csv')]

## Key Findings

- Average airfare varied substantially across the Top 10 SEA destinations.
- In December 2025, LAS had the lowest average fare at approximately $170.65.
- ANC had the highest December 2025 average fare at approximately $282.22.
- The 2022–2024 values are based on DB1B Q4 data, while the 2025 value is based on December DB1C data.
- Because the reporting periods and survey methodologies differ, the airfare values are used descriptively rather than interpreted as a perfectly comparable four-year fare trend.

The processed airfare datasets produced in this notebook are used in the master dataset preparation and Power BI dashboard.